# 03 — Découpage en passages et vocabulaire

**Projet** : Assistant documentaire RAG orchestré avec LangChain (LCEL) — chaîne prompt + retriever + citations

## Objectifs pédagogiques

1. Découper un document en passages dont les offsets pointent réellement sur le texte annoncé.
1. Observer le filtrage des mots vides et son effet sur le vocabulaire.
1. Mesurer l'effet de la taille des passages sur le rappel@5, au lieu de la choisir au ressenti.

**Objectifs transverses du dépôt**

- Découper un corpus en passages : pourquoi le chevauchement existe et comment mesurer son effet sur le rappel.
- Comprendre BM25 (saturation, normalisation de longueur, IDF) en l'implémentant, plutôt qu'en l'appelant.
- Évaluer un système de recherche par question puis moyenne : recall@k, MRR, nDCG@k et ce que chacun ignore.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.config_access import node  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet journalise au niveau INFO ; un notebook noierait ses tableaux sous ces lignes.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` -----------------------------------
# Les notebooks travaillent sur un corpus réduit (64 documents) quand `data/raw` est
# vide : l'exécution reste rapide. Si le corpus de référence a été généré par `make data`, il est
# utilisé tel quel — les valeurs absolues d'un notebook ne sont pas celles de la référence.
NB_DOCUMENTS = 64

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_DOCUMENTS}",
                "seed=42",
                "log_level=WARNING",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks lisent le corpus du projet et écrivent leurs artefacts dans `outputs/notebooks` :
# ils ne doivent jamais écraser ceux produits par `make train` / `make evaluate`.
NB_PATHS = ProjectPaths(
    root=PROJECT_ROOT,
    data_dir=PATHS.data_dir,
    artifacts_dir=PROJECT_ROOT / "outputs" / "notebooks" / "artifacts",
).ensure()

CHUNKING = dict(node(CONFIG, "preprocessing").get("chunking", {}))

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (plancher contractuel : 0.6)")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(
    f"Découpage         : {CHUNKING.get('max_tokens')} tokens, "
    f"chevauchement {CHUNKING.get('overlap_tokens')}"
)
print(f"Sorties notebook  : {NB_PATHS.artifacts_dir.relative_to(PROJECT_ROOT)}")

## 1. Du texte brut aux passages

Le découpage est la décision la plus structurante d'un système de recherche : un passage trop court perd le contexte, un passage trop long dilue le signal. Ici le découpage se fait **par phrase** et conserve ses positions : un passage doit pouvoir être affiché tel quel dans une réponse sourcée.

In [ ]:
from src.data.generators import SyntheticCorpusGenerator
from src.data.loaders import TextCorpusLoader

LOADER = TextCorpusLoader(PATHS, formats=CONFIG.data.formats)
if LOADER.documents_path.exists():
    DOCUMENTS = LOADER.load_documents()
    QUERIES = LOADER.load_queries()
    METADATA = LOADER.load_metadata()
    print(f"Corpus lu depuis data/raw : {len(DOCUMENTS)} documents, {len(QUERIES)} questions")
else:
    # Un clone frais n'a pas encore de données : le notebook reste exécutable. Le corpus réduit est
    # persisté pour que les pipelines (train, evaluate) trouvent les mêmes fichiers que le
    # notebook ; `make data` régénère le corpus de référence.
    bundle = SyntheticCorpusGenerator.from_config(CONFIG.data).generate()
    DOCUMENTS, QUERIES, METADATA = bundle.documents, bundle.queries, bundle.metadata
    LOADER.save_documents(DOCUMENTS)
    LOADER.save_queries(QUERIES)
    LOADER.save_metadata(METADATA)
    print(f"data/raw absent : corpus de {len(DOCUMENTS)} documents généré et persisté")

CALIBRATION = QUERIES[QUERIES["split"] == "calibration"].reset_index(drop=True)
VALIDATION = QUERIES[QUERIES["split"] == "val"].reset_index(drop=True)
TEST = QUERIES[QUERIES["split"] == "test"].reset_index(drop=True)
ANSWERABLE = QUERIES[QUERIES["answer_type"] != "unanswerable"].reset_index(drop=True)
# Sous-ensemble utilisé pour **régler** quoi que ce soit : la validation seule. Le test ne sert
# qu'à juger, jamais à choisir (y compris dans un notebook).
VAL_ANSWERABLE = VALIDATION[VALIDATION["answer_type"] != "unanswerable"].reset_index(drop=True)
print(
    "splits :",
    {
        name: len(frame)
        for name, frame in (("calibration", CALIBRATION), ("val", VALIDATION), ("test", TEST))
    },
)

In [ ]:
from src.preprocessing.transformers import DocumentChunker, StopWordFilter, tokenize

sample = DOCUMENTS.iloc[0]
chunker = DocumentChunker(**CHUNKING)
chunks = chunker.chunk_document(str(sample["doc_id"]), str(sample["text"]))
print(f"document {sample['doc_id']} : {sample['n_tokens']} tokens -> {len(chunks)} passages")
for chunk in chunks:
    segment = str(sample["text"])[chunk.start_char : chunk.end_char]
    verdict = "offsets exacts" if segment.strip() == chunk.text.strip() else "OFFSETS INCOHÉRENTS"
    print(
        f"  {chunk.chunk_id} [{chunk.start_char}:{chunk.end_char}] "
        f"{chunk.n_tokens:>3} tokens — {verdict}"
    )

In [ ]:
print("texte du premier passage, tel qu'il sera cité :")
print(chunks[0].text[:400], "…")
print()
print("tokens du même passage :", tokenize(chunks[0].text)[:15], "…")

## 2. Tokénisation et mots vides

La recherche lexicale raisonne sur des termes. Retirer les mots vides français est ce qui rend le score discriminant : sans cela, « de », « la » et « pour » dominent et deux textes sans rapport paraissent proches.

In [ ]:
raw_tokens = tokenize("Le solde de congés payés est de 25 jours ouvrables par an.")
filtered = StopWordFilter().transform(raw_tokens)
print("tokens bruts   :", raw_tokens)
print("tokens filtrés :", filtered)
print(f"{len(raw_tokens) - len(filtered)} mots vides retirés sur {len(raw_tokens)}")

## 3. Le corpus découpé

On applique le pipeline de prétraitement au corpus entier, puis on regarde ce qui est réellement indexé : nombre de passages, longueur moyenne et distribution.

In [ ]:
from src.preprocessing.pipelines import TextPreprocessor

PREPROCESSOR = TextPreprocessor.from_config(node(CONFIG, "preprocessing"))
CHUNKS = PREPROCESSOR.prepare_corpus(DOCUMENTS)
print(f"{len(DOCUMENTS)} documents -> {len(CHUNKS)} passages indexés")
print(f"longueur moyenne d'un passage : {CHUNKS['n_tokens'].mean():.1f} tokens")
display(CHUNKS.head(3)[["chunk_id", "doc_id", "chunk_index", "n_tokens"]])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
CHUNKS["n_tokens"].plot.hist(ax=axes[0], bins=25, title="Longueur des passages (tokens)")
CHUNKS.groupby("doc_id").size().plot.hist(
    ax=axes[1], bins=15, title="Passages produits par document"
)
axes[0].set_xlabel("tokens")
axes[1].set_xlabel("passages")
plt.tight_layout()
plt.show()

## 4. Effet de la taille des passages sur le rappel

Chaque taille testée re-découpe et ré-indexe le corpus. C'est exactement l'expérience à mener avant de choisir un découpage : elle coûte quelques secondes avec un index lexical, et elle remplace une intuition par une mesure.

In [ ]:
import numpy as np

from src.models import build_model
from src.training.losses_metrics import retrieval_metrics

gold = [{item for item in str(ids).split(",") if item} for ids in VAL_ANSWERABLE["gold_doc_ids"]]

rows = []
for size in [40, 60, 110, 180]:
    variant_config = CONFIG.model_dump()
    variant_config["preprocessing"]["chunking"] = {
        "max_tokens": size,
        "overlap_tokens": max(size // 4, 10),
    }
    variant = build_model(variant_config)
    fit_result = variant.fit(DOCUMENTS)
    print(
        f"taille {size:>4} tokens -> {fit_result.n_chunks} passages "
        f"(vocabulaire {fit_result.extra.get('vocabulary_size')} termes)"
    )
    rankings = [
        [passage.doc_id for passage in variant.retrieve(str(question), 5)]
        for question in VAL_ANSWERABLE["question"]
    ]
    rows.append(
        {
            "taille": size,
            "passages": fit_result.n_chunks,
            "vocabulaire": fit_result.extra.get("vocabulary_size", float("nan")),
            "recall_at_5": retrieval_metrics(rankings, gold, ks=(5,))["recall_at_5"],
        }
    )

benchmark = pd.DataFrame(rows)
n_questions = len(VAL_ANSWERABLE)
# Un rappel mesuré sur ~50 questions porte son propre bruit : l'intervalle de confiance évite de
# conclure sur un écart de deux points. C'est la différence entre un notebook et une croyance.
benchmark["intervalle_95"] = 1.96 * np.sqrt(
    benchmark["recall_at_5"] * (1.0 - benchmark["recall_at_5"]) / n_questions
)
print(
    f"questions de validation utilisées : {n_questions} "
    f"(longueur moyenne d'un document : {DOCUMENTS['n_tokens'].mean():.0f} tokens)"
)
benchmark.round(4)

In [ ]:
best = benchmark.loc[benchmark["recall_at_5"].idxmax()]
configured = benchmark.loc[benchmark["taille"] == 110]
print(
    f"meilleure taille observée : {int(best['taille'])} tokens "
    f"(rappel@5 = {best['recall_at_5']:.3f} ± {best['intervalle_95']:.3f})"
)
if len(configured):
    gap = float(best["recall_at_5"] - configured["recall_at_5"].iloc[0])
    verdict = "au-delà du bruit" if abs(gap) > float(best["intervalle_95"]) else "dans le bruit"
    print(f"écart au découpage configuré ({110} tokens) : {gap:+.3f} -> {verdict}")
plateau = benchmark.groupby("passages")["taille"].apply(list)
for passages, sizes in plateau.items():
    if len(sizes) > 1:
        print(
            f"tailles {sizes} : mêmes {passages} passages indexés, donc même mesure "
            "— au-delà de la longueur d'un document, un découpage plus grand ne change rien"
        )

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(benchmark["taille"], benchmark["passages"], marker="o")
axes[0].set(title="Passages indexés", xlabel="taille d'un passage (tokens)", ylabel="passages")
axes[1].errorbar(
    benchmark["taille"],
    benchmark["recall_at_5"],
    yerr=benchmark["intervalle_95"],
    marker="o",
    color="darkgreen",
    capsize=4,
)
axes[1].set(
    title="Rappel@5 et son intervalle à 95 %",
    xlabel="taille d'un passage (tokens)",
    ylabel="recall@5",
)
plt.tight_layout()
plt.show()

**Ce qu'il faut retenir**

- Le résultat est un **plateau**, pas un optimum : dès qu'un passage couvre tout le document, agrandir la taille ne change plus ni l'index ni la mesure.
- Toute différence inférieure à l'intervalle de confiance affiché est du bruit : sur quelques dizaines de questions, décider sur un écart de deux points revient à tirer à pile ou face. Il faut plus de questions, pas plus d'intuition.
- Chaque taille re-indexe le corpus : l'expérience n'est acceptable que parce que l'index lexical se construit en quelques secondes — un argument de plus pour commencer par une baseline lexicale.
- Le nombre de passages indexés fixe la latence et la mémoire : le choix du découpage est un arbitrage mesuré, pas une constante recopiée.